<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/CyberShield-Threat-Detector/blob/main/URL-Phishing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install -q xgboost lightgbm imbalanced-learn tldextract
import tldextract
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV , RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import joblib
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV , RandomizedSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import re
from sklearn.utils.class_weight import compute_sample_weight
import joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 2.2 MB/s eta 0:00:00


In [13]:
# === Use this  or Use direct download from google server
# from google.colab import files
# uploaded = files.upload()


# Getting the data straight from Kaggle so we don't have to manually upload it every time.
import os
os.environ['KAGGLE_API_TOKEN'] = ''
!pip install -q kaggle
!kaggle datasets download -d sid321axn/malicious-urls-dataset
import zipfile
import random
with zipfile.ZipFile('/content/malicious-urls-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

Dataset URL: https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset
License(s): CC0-1.0
malicious-urls-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [14]:
from google.colab import files
uploaded = files.upload()

Saving url_dataset.csv to url_dataset (1).csv


In [15]:
manual_df = pd.read_csv(r'url_dataset.csv')
manual_df.head(10)

,url,type
0,https://google.com,benign
1,https://microsoft.com,benign
2,https://amazon.com,benign
3,https://apple.com,benign
4,https://youtube.com,benign
5,https://wikipedia.org,benign
6,https://linkedin.com,benign
7,https://instagram.com,benign
8,https://facebook.com,benign
9,https://netflix.com,benign


##  Load Dataset

Raw file has 2 columns — `url` and `type`. Keeping original df clean before adding anything to it.

In [16]:
malicious_df = pd.read_csv(r'malicious_phish.csv')
historic_df = malicious_df.copy()
malicious_df.head(10)

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement
5,http://buzzfil.net/m/show-art/ils-etaient-loin...,benign
6,espn.go.com/nba/player/_/id/3457/brandon-rush,benign
7,yourbittorrent.com/?q=anthony-hamilton-soulife,benign
8,http://www.pashminaonline.com/pure-pashminas,defacement
9,allmusic.com/album/crazy-from-the-heat-r16990,benign


In [10]:
historic_df.dtypes

,0
url,object
type,object


## Data Inspection

Checking class distribution, data quality, and duplicates before touching anything. Dataset is heavily imbalanced — benign is 66% of all rows. 10k+ duplicates expected in a real-world scraped URL dataset.

In [17]:
# Initial sanity check: Inspecting class imbalance and data quality before feature engineering
print(f"Shape: {malicious_df.shape}")
print("\nLabel distribution:")
print(malicious_df['type'].value_counts())
print('\nMissing Values:')
print(malicious_df.isnull().sum())
print(f"\nDuplicates: {malicious_df.duplicated().sum()}")

Shape: (651191, 2)

Label distribution:
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Missing Values:
url     0
type    0
dtype: int64

Duplicates: 10066


In [18]:
# Dropping exact repeated URLs to prevent model bias and avoid data leakage between train/test sets
malicious_df.drop_duplicates(inplace=True)

malicious_df = malicious_df.reset_index(drop=True)
print(f'After dedup+reset: {malicious_df.shape}')


After dedup+reset: (641125, 2)


In [19]:
print('Duplicate:',malicious_df.duplicated().sum())

Duplicate: 0


In [20]:
df = pd.concat([malicious_df,manual_df],ignore_index=True)
print("After Merge:",df.shape,'\n\n')

print("Excat duplicate rows:",df.duplicated().sum())


After Merge: (641173, 2) 


Excat duplicate rows: 1


In [25]:
df[df.duplicated(keep=False)]

,url,type
641168,https://drive.google.com/uc?export=download&id...,malware
641169,https://drive.google.com/uc?export=download&id...,malware


In [27]:
df = df.drop_duplicates().reset_index(drop=True)
print("Rechecking Duplicate:",df.duplicated().sum())

Rechecking Duplicate: 0


In [ ]:
# normalize URLs: 'google.com' → 'http://google.com'
# ensures consistent feature extraction across all classes
def normalize_url(url):
    url = str(url).strip()
    if not re.match(r'^https?://', url, re.IGNORECASE):
        url = 'http://' + url
    return url

malicious_df['url'] = malicious_df['url'].apply(normalize_url)
print('Normalized. Sample:')
print(malicious_df['url'].head(5).tolist())


In [ ]:
malicious_df.tail()

## Label Encoding

Manual mapping instead of sklearn's LabelEncoder — this way we control which number maps to which class. Matters when reading the confusion matrix later.

In [ ]:
# Converting text classes to numeric targets since ML algorithms (like XGBoost) can't process strings
label_map = {
    'benign': 0,
    'phishing':1,
    'defacement':2,
    'malware':3,
}
malicious_df['label'] = malicious_df['type'].map(label_map)
# Double-checking the mapping and ensuring no unmapped garbage values slipped through
print(malicious_df[['type','label']].drop_duplicates().sort_values('label'))
print(f"\nNull Labels:{malicious_df['label'].isnull().sum()}")

malicious_df = malicious_df.reset_index(drop=True)
print(f'After dedup+reset: {malicious_df.shape}')


### Feature Engineering
This is the NLP part — treating the raw URL string as text and extracting signals from it. I've grouped the extracted features into 4 main categories:

* **Statistical:** `url_length`, `hyphen_count`, `dot_count`, `slash_count`, `special_char_count`
  *(Catches unusually long or complex structures)*
* **Structural:**  `has_at`, `has_double_slash`
  *(Catches protocol tricks and redirect abuse)*
* **Domain/Path:** `subdomain_count`, `query_length`, `path_length`
  *(Catches deep hiding subdomains and suspiciously long parameters)*
* **Semantic:** `phishing_keyword`, `has_ip`
  *(Catches social engineering words and raw IP-based routing)*

In [ ]:
def extract_feature(url):
    url = str(url)
    # strip protocol before feature extraction
    url_clean = re.sub(r'^https?://', '', url, flags=re.IGNORECASE)
    f = {}
    f['url_length']         = len(url_clean)
    f['hyphen_count']       = url_clean.count('-')
    f['dot_count']          = url_clean.count('.')
    f['slash_count']        = url_clean.count('/')
    f['special_char_count'] = len(re.findall(r'[@?=&%]', url_clean))
    f['has_at']             = int('@' in url_clean)
    f['has_double_slash']   = int('//' in url_clean)
    domain_only             = url_clean.split('/')[0]
    f['subdomain_count']    = max(domain_only.count('.') - 1, 0)
    if '?' in url_clean:
        bq, aq              = url_clean.split('?', 1)
        f['query_length']   = len(aq)
        f['path_length']    = len(bq.split('/', 1)[1]) if '/' in bq else 0
    else:
        f['query_length']   = 0
        f['path_length']    = len(url_clean.split('/', 1)[1]) if '/' in url_clean else 0
    kw = r'login|verify|secure|bank|paypal|update|account|signin'
    f['phishing_keyword']   = int(bool(re.search(kw, url_clean, re.IGNORECASE)))
    ip_pat = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
    f['has_ip']             = int(bool(re.search(ip_pat, url_clean)))
    return f

feature_list = malicious_df['url'].apply(extract_feature).tolist()
features_df  = pd.DataFrame(feature_list)

# features_df has same length and order as malicious_df (both reset_index'd)
FEAT_COLS = list(features_df.columns)
for col in FEAT_COLS:
    malicious_df[col] = features_df[col].values

print(f'features_df shape: {features_df.shape}')
print(f'malicious_df columns: {list(malicious_df.columns)}')
print(features_df.head())


In [ ]:
# Quick sanity check: Let's test our extraction logic on a few custom URLs
# to make sure the regex and counters are actually catching the right signals.
test_urls = [
    'https://google.com',
    'https://paypal-verify.xyz/login',
    'http://site.com//redirect/evil',
    'http://192.168.1.1/payload'
]

# Loop through our test cases and print out a few critical features to verify
for u in test_urls:
    f = extract_feature(u)
    print(f'URL: {u}')
    print(f'  slash={f["slash_count"]}  double_slash={f["has_double_slash"]}  ip={f["has_ip"]}  keyword={f["phishing_keyword"]}')

In [ ]:
feature_cols = [
    'url_length', 'hyphen_count', 'dot_count', 'slash_count',
    'special_char_count', 'has_at', 'has_double_slash',
    'subdomain_count', 'query_length', 'path_length',
    'phishing_keyword', 'has_ip'
]
print('feature_cols:', feature_cols)


In [ ]:
malicious_df.head()

###Exploratory Data Analysis (EDA)

*Note: Doing EDA after Feature Engineering is an intentional choice here.*

Raw URL strings don't give us much to plot or analyze directly. By extracting our numerical features first, we can now actually visualize things like URL length distributions, structural tricks, and hidden patterns across different classes. Let's see what the data tells us!

### Class Distribution & The Real-World Catch

* **Initial thought:** "66% benign — imbalanced but not extreme. `class_weight='balanced'` in the model will handle it."
* **The Reality:** Testing proved this assumption wrong. While `class_weight` fixes the *numerical* imbalance, it cannot fix *dataset bias*.
* **The Issue:** The 66% benign URLs in this Kaggle dataset lack representation of short, bare domains (like `google.com`). Forcing the model to prioritize the minority classes just made it over-sensitive, leading to false positives on unseen safe domains.

In [ ]:
# Visualizing the target distribution to highlight the severe class imbalance.
# The bar chart shows absolute volume, while the donut chart exposes the percentage skew (justifying why we care more about F1/Recall than pure Accuracy)
fig,axes = plt.subplots(1,2,figsize=(14,5))
counts = malicious_df['type'].value_counts()
bar_colors = ['#1abc9c', '#f39c12', '#7f8c8d', '#e74c3c']
bars = axes[0].bar(counts.index,counts.values,color=bar_colors,edgecolor="none",width=0.55,alpha=0.9)
axes[0].set_ylabel('Frequency',fontsize=10)
axes[0].tick_params(axis='x', rotation=10)

for i in bars:
  yval = i.get_height()
  axes[0].text(i.get_x()+ i.get_width()/2.0,yval+5000,f"{yval:,}",
               ha='center',va = 'bottom',fontsize=10,fontweight='bold')


wadges,texts,autotexts = axes[1].pie(counts.values,labels=counts.index
                                     ,autopct="%1.1f%%",startangle=140,colors = bar_colors,wedgeprops=dict(width=0.6,edgecolor='white',linewidth=2)
                                     )


plt.setp(autotexts, size=9, weight="bold", color="white")
plt.setp(texts, size=10)
axes[1].set_title('Class Proportion Split', fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

**What this tells us:**

- **`benign` = 66.8%** (428,080 URLs) — majority class. A lazy model that predicts everything as benign would get 66.8% accuracy for free — which is why accuracy is a useless metric here.
- **`defacement` = 14.9%**, **`phishing` = 14.7%** — roughly equal minority classes.
- **`malware` = 3.7%** (23,645 URLs) — rarest and most dangerous. Our XGBoost misses 12% of malware (recall = 88%). In a real security system, that 12% represents actual attacks getting through.

> **Bottom line:** Macro F1 is the only honest metric for this dataset. Accuracy is inflated by the 66.8% benign dominance and means nothing here.

In [ ]:
malicious_df['tld'] = malicious_df['url'].apply(lambda x: tldextract.extract(str(x)).suffix)

malware_blanks = malicious_df[(malicious_df['type'] == 'malware') & ((malicious_df['tld'].isna()) | (malicious_df['tld'] == ''))]
print(malware_blanks['url'].head(20).tolist())

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
colors = {'benign': 'steelblue', 'phishing': 'tomato', 'defacement': 'orange', 'malware': 'crimson'}

for ax, label in zip(axes.flatten(), ['benign', 'phishing', 'defacement', 'malware']):
    top_tld = malicious_df[malicious_df['type'] == label]['tld'].value_counts().head(10)
    ax.barh(top_tld.index[::-1], top_tld.values[::-1], color=colors[label])
    ax.set_title(f"Top 10 TLDs - {label}")
    ax.set_xlabel('Frequency')
plt.suptitle('Top 10 TLDs Per Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Convert blank or null TLDs into IP_Address category
malicious_df['tld'] = malicious_df['tld'].replace('', "IP_Address")
malicious_df['tld'] = malicious_df['tld'].fillna('IP_Address')

print(malicious_df['tld'].value_counts().head(10))

In [ ]:
# Standardizing missing TLDs: URLs without a domain extension are typically direct IP addresses.
# Explicitly grouping them as 'IP_Address' creates a powerful signal for the model to detect evasive malware
malicious_df['tld'] = malicious_df['tld'].astype(str).str.strip()
malicious_df.loc[malicious_df['tld'] == '', 'tld'] = 'IP_Address'
malicious_df.loc[malicious_df['tld'] == 'nan', 'tld'] = 'IP_Address'
print(malicious_df['tld'].value_counts().head(5))

# Visualizing the cleaned TLD distribution to verify how heavily 'IP_Address' dominates the malicious classes
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

for ax, label in zip(axes.flatten(), ['benign', 'phishing', 'defacement', 'malware']):
    top_tld = malicious_df[malicious_df['type'] == label]['tld'].value_counts().head(10)
    ax.barh(top_tld.index[::-1], top_tld.values[::-1], color=colors[label])
    ax.set_title(f"Top 10 TLDs - {label}")
    ax.set_xlabel('Frequency')
plt.suptitle('Top 10 TLDs Per Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 📊 TLD Distribution Insights

*   **Benign & Phishing:** Both heavily dominated by `.com`, `.org`, and `.net`. Phishers intentionally mimic legitimate TLDs, which perfectly explains why Phishing is the hardest class to isolate (lowest recall).
*   **Defacement:** Spread across country-level TLDs (`.de`, `.nl`, `.it`, `.com.br`). Attackers randomly target vulnerable CMS platforms globally, regardless of hosting origin.
*   **Malware:** `IP_Address` is the #1 entry (~12k URLs). Raw IPs are a near-perfect malware signal since legitimate services rarely use them. The unexpected spike in `.jp` suggests compromised Japanese servers acting as payload hosts.

> **Note:** A missing TLD (`IP_Address`) is a high-confidence malware indicator, while Phishing's heavy reliance on `.com` allows it to easily blend into normal traffic.

In [ ]:
# Add the label column to features_df just for this correlation plot
features_df['label'] = malicious_df['label']

# Now plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    features_df[feature_cols + ['label']].corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)
plt.title("Feature Correlation Heatmap")
plt.show()

###Correlation Insights

Reading the actual values from the heatmap:

*   **The MVP Signal:** `has_ip` ↔ `label` = **0.37** — The strongest individual feature correlated with malicious labels.
*   **Redundant Pair 1:** `url_length` ↔ `query_length` = **0.74** — Almost identical information. Longer URLs naturally mean longer queries.
*   **Redundant Pair 2:** `dot_count` ↔ `subdomain_count` = **0.76** — More dots usually just map to more subdomains.
*   **Moderate Overlap:** `slash_count` ↔ `path_length` = **0.57** — Moderate structural overlap.

> **Note:** `has_ip` is the most powerful standalone feature. The correlated pairs (`url_length`/`query_length`, `dot_count`/`subdomain_count`) represent redundant information. Dropping one feature from these pairs will make our production model leaner without losing predictive power.

### URL Length Distribution
Malware and defacement URLs tend to be longer — more path segments, longer query strings. Phishing is surprisingly short — quick redirect links.

In [ ]:
# Investigating if URL length is a strong differentiator. Malicious URLs are often unusually long to hide payloads or excessive parameters.
# Note: Clipping lengths at 300 characters strictly for visualization to prevent extreme outliers from squashing the charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = {0: 'steelblue', 1: 'tomato', 2: 'orange', 3: 'crimson'}

for name, lbl in label_map.items():
    data = malicious_df[malicious_df['label'] == lbl]['url_length'].clip(upper=300)
    axes[0].hist(data, bins=60, alpha=0.5, color=colors[lbl], label=name)

axes[0].set_xlabel("URL Length")
axes[0].set_ylabel('Frequency')
axes[0].set_title("URL Length")
axes[0].legend()

# Using a Boxplot to clearly see the medians and the spread of outliers across classes
box_data = [
    malicious_df[malicious_df['label'] == l]['url_length'].clip(upper=300).values
    for l in label_map.values()
]
bp = axes[1].boxplot(box_data, labels=['benign', 'phishing', 'defacement', 'malware'],
                     patch_artist=True)

for box, color in zip(bp['boxes'], colors.values()):
    box.set_facecolor(color)
    box.set_alpha(0.7)

axes[1].set_title('URL Length')
axes[1].set_ylabel('URL Length')

plt.tight_layout()
plt.show()

# Printing exact summary statistics to back up our visual findings mathematically
print(malicious_df.groupby('label')['url_length'].describe().round(1))

###URL Length Distribution: A Counterintuitive Discovery

Extracting insights from the boxplot medians before feeding this to the model:

*   **Phishing (Median ~35 chars):** Surprisingly, this is the *shortest* class. Attackers intentionally keep phishing URLs short and clean (e.g., `br-icloud.com.br`) to bypass length-based security filters and look trustworthy.
*   **Malware (Median ~45 chars):** Short median, but heavy outliers on the right. This aligns perfectly with our previous findings: malware ranges from very short raw IPs (`192.168.1.1/file`) to extremely long payload paths.
*   **Benign (Median ~50 chars):** Sits right in the middle, representing a natural, wide spread of standard domains and content links.
*   **Defacement (Median ~85 chars):** The longest class by far. This makes sense as compromised CMS platforms auto-generate deeply nested URLs (e.g., `/index.php?option=com_content&view=article...`).

> **Why this matters for Training:** Common sense assumes phishing URLs are long and suspicious-looking. The data proves the exact opposite. If our model learns a simple rule like "longer URL = more malicious," it will completely fail to detect modern, short phishing domains.

### Numerical Feature Distribution Per Class
Boxplots only for continuous numerical features — binary flags are handled separately below. Features with clearly different medians across classes are the most useful for the model.

In [ ]:
# Analyzing structural components (dots, slashes, special chars) across classes.
# Phishing and Malware URLs often abuse these to create convincing fake domains or hide complex payloads
count_features = ['hyphen_count', 'dot_count', 'slash_count',
                  'special_char_count', 'query_length', 'path_length']

for col in count_features:
  malicious_df[col] = pd.to_numeric(malicious_df[col],errors='coerce')

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for ax, feat in zip(axes, count_features):
    cap = malicious_df[feat].quantile(0.99)
    box_data = [
        malicious_df[malicious_df['label'] == l][feat].clip(upper=cap)
        for l in label_map.values()
    ]
    bp = ax.boxplot(box_data, labels=list(label_map.keys()), patch_artist=True)
    for box, color in zip(bp['boxes'], bar_colors):
        box.set_facecolor(color)
        box.set_alpha(0.7)

    ax.set_title(feat)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

###Feature Analysis (Boxplots)

Extracting signals from individual features before training:

*   **`special_char_count` & `query_length`:** Defacement completely dominates here. The heavy use of `?`, `=`, and `&` (median query length ~20) is a massive footprint of compromised CMS platforms (like WordPress). Other classes stay near zero.
*   **`dot_count`:** Malware pulls ahead with a higher median (3). This perfectly aligns with our `IP_Address` finding — raw IPs (`192.168.1.1`) inherently contain exactly 3 dots.
*   **`hyphen_count`:** Malware is dead flat at zero (simple structural delivery payloads). Benign URLs show the most variance because legitimate blogs and articles frequently use hyphens for SEO (`/how-to-guide`).
*   **`path_length`:** Benign URLs actually have the longest median paths here, reflecting deep, legitimate website structures. Phishing stays minimal, intentionally mimicking clean, top-level domains.
*   **`slash_count`:** Very similar spread across all classes. This feature will likely add minimal discriminative power to the model on its own.

> **Key Takeaway for the Model:** `query_length` and `special_char_count` are powerful, isolated signals for catching Defacement. Phishing, however, scores near-zero across almost all numerical features. It deliberately mimics the clean structure of Benign URLs, which explains why Phishing is historically the hardest class for ML models to isolate.

### Binary Feature Distribution Per Class
Percentage of URLs in each class where the binary flag = 1. `has_ip` is the strongest signal — malware uses IP addresses heavily to bypass DNS. Boxplot would just show flat lines here — percentage bars give the real insight.

In [ ]:
# Binary features
binary_features = ['has_ip','has_at','has_double_slash',
                   'phishing_keyword',]
# Colors for each class
colors = ['steelblue', 'tomato', 'orange', 'crimson']
# Create subplots (share y-axis)
fig, axes = plt.subplots(
    2, 2,
    figsize=(20, 9),
    sharey=True
)
axes = axes.flatten()
for ax, col in zip(axes, binary_features):
    # Percentage of URLs having feature = 1
    pct = malicious_df.groupby('type')[col].mean() * 100
    pct.index = pct.index.map(label_map)
    # Bar plot
    bars = ax.bar(pct.index,
        pct.values,color=colors,edgecolor='black',alpha=0.85)
    # Title
    ax.set_title(col, fontsize=13, fontweight='bold')
    # Labels
    ax.set_ylabel('% URLs with feature = 1')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.tick_params(axis='x', rotation=15)
    # Percentage labels
    for bar, val in zip(bars, pct.values):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            min(val + 1.5, 98),
            f'{val:.1f}%',
            ha='center',va='bottom',fontsize=10,fontweight='bold')
axes[-1].axis('off')
plt.suptitle(
    'Binary Feature Distribution by Class (%)',
    fontsize=16,
    fontweight='bold'
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

###Binary Features: The True Signals vs. The Noise

Reading the distribution charts (where X-axis labels roughly map to our encoded classes: 0=Benign, 1=Phishing, 2=Defacement, 3=Malware):

*   **`has_ip` (The MVP):** Hits **49.8%** on Malware (Class 3) while staying near 0% for everything else. Almost half of all malware URLs in this dataset use raw IP addresses. Because legitimate services almost never do this, it serves as a near-perfect malware signal.
*   **`phishing_keyword` (The Disappointment):** Present in only **10.7%** of Phishing URLs compared to 6.4% in Benign URLs. A mere 4.3% difference makes it highly unreliable as a standalone feature. Modern attackers deliberately avoid obvious words like `login` or `secure` to bypass basic filters.
*   **`has_double_slash` & `has_at` (The Noise):** Both features score **≤ 1.0%** across all classes. They are extremely rare and provide almost zero discriminative value for this specific dataset.

> **Key Takeaway for the Model:** `has_ip` is the only binary feature doing heavy lifting here. The weakness of `phishing_keyword` directly explains why Phishing is our hardest class to catch—it simply doesn't rely on the classic text "tricks" we expect it to.

In [ ]:
malicious_df.head()

## Machine Learning

Features ready — time to train. TLD is a string column so it gets frequency-encoded before going into the model. `stratify=y` keeps class ratios same in train and test.

In [ ]:
# use malicious_df — features were merged into it after extract_feature
X = malicious_df[feature_cols].copy()
y = malicious_df['label']

# tld_series with positional index matching malicious_df
tld_series = malicious_df['url'].apply(
    lambda u: tldextract.extract(u).suffix or 'no_tld'
).reset_index(drop=True)

print(f'X shape: {X.shape}  y shape: {y.shape}')


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size=0.2,random_state=42,stratify=y
)

In [ ]:
# use positional indexing to avoid index mismatch
train_pos = list(X_train.index)
test_pos  = list(X_test.index)

tld_train = tld_series.iloc[train_pos]
tld_freq  = tld_train.value_counts(normalize=True).to_dict()

X_train = X_train.fillna(0).copy()
X_test  = X_test.fillna(0).copy()

X_train['tld_encoded'] = tld_series.iloc[train_pos].map(tld_freq).fillna(0).values
X_test['tld_encoded']  = tld_series.iloc[test_pos].map(tld_freq).fillna(0).values

print(f'Train TLD NaN: {X_train["tld_encoded"].isna().sum()}')
print(f'Test  TLD NaN: {X_test["tld_encoded"].isna().sum()}')
print(f'Final X_train shape: {X_train.shape}')


In [ ]:
X_train.isnull().sum()

### Logistic Regression
Baseline linear model. Good for understanding which features have linear relationships with the target. Phishing class (1) recall is very low — features aren't linearly separable here.

In [ ]:
log_reg = LogisticRegression(multi_class='multinomial',max_iter=1000)
param_grid = {
    'C': [0.01,0.1,1,10],
    "penalty": ['l2'],
    'solver': ['lbfgs']
}

grid_search = GridSearchCV(
    estimator=log_reg,param_grid=param_grid,cv=3,
    verbose=2,n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("\n--- GridSearch Complete ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")
best_log_reg_model = grid_search.best_estimator_

In [ ]:
lgr_model = Pipeline([
    ('scaler',StandardScaler(with_mean=False)),
    ('classifier', LogisticRegression(
        C=10,
        solver='lbfgs',
        # multi_class='multinomial',
        max_iter=1000
    ))
])
lgr_model.fit(X_train, y_train)
y_pred_log = lgr_model.predict(X_test)

accuracy_lgr = accuracy_score(y_test,y_pred_log)
print(f"Model Accuracy : {accuracy_lgr}")

print("\n---Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_log))

print('\n--Classification Report:')
print(classification_report(y_test,y_pred_log))

###Logistic Regression Results: The Linear Limitation

*   **The Accuracy Illusion (73%):** The overall accuracy looks decent, but it's a trap. The model achieved this simply by predicting almost everything as Benign (Class 0), which has a massive 95% recall.
*   **The Phishing Failure (Recall = 12%):** This is exactly what we predicted in the EDA. Because Phishing deliberately mimics the structure of Benign URLs, a linear boundary cannot separate them. A staggering 16,001 phishing URLs were incorrectly classified as Safe (Benign).
*   **Malware Catch Rate (Recall = 49%):** While Precision is high (95% — meaning when it flags malware, it's usually right), it completely misses over half the malware (2,290 URLs slipped through as Benign). It likely only caught the obvious ones, like raw IP addresses.
*   **Macro F1-Score (0.55):** This is the true reflection of our baseline model. It performs poorly across the minority classes.

>**Verdict:** The data is clearly non-linear. This baseline proves we can't use linear models here — we strictly need tree-based architectures (like Random Forest or XGBoost) to catch these complex relationships.

### Naive Bayes (GridSearch)

**The Probabilistic Baseline.** We are including Naive Bayes strictly for completeness. By definition, this algorithm assumes that all features are completely independent of each other (hence the name "Naive").

However, our EDA heatmap already proved this assumption is false for our dataset—features like `url_length` and `query_length` are heavily correlated (0.74). Because it ignores these structural relationships, we fully expect this model to underperform. Let's look at the results to confirm.

In [ ]:
nb_pipeline = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier',MultinomialNB())
])

param_grid = {
    'classifier__alpha':[0.01,0.1,0.5,1.0,5.0,10.0]
}

nb_grid = GridSearchCV(nb_pipeline,param_grid,cv=3,scoring='accuracy',n_jobs=-1,verbose=2)
nb_grid.fit(X_train,y_train)

print(f"Best Paramter : {nb_grid.best_params_}")

In [ ]:
naive_b_model = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier',MultinomialNB(alpha=0.1))
])

naive_b_model.fit(X_train,y_train)
y_pred_nb = naive_b_model.predict(X_test)

accuracy_nb = accuracy_score(y_test,y_pred_nb)

print('\n---Confusion Matrix')
print(confusion_matrix(y_test,y_pred_nb))

print('\n--Classification Report')
print(classification_report(y_test,y_pred_nb))

### Naive Bayes Results

* **Accuracy Drop (72%):** Performed slightly worse than the Logistic Regression baseline.
* **The Independence Flaw:** NB mathematically assumes all features are independent. We already saw in our EDA heatmap that features like `url_length` and `query_length` are heavily correlated (0.74). This false assumption confused the model.
* **Phishing & Malware (Still Failing):** Phishing recall is only 17% (it completely missed 15,176 phishing URLs, calling them Safe). Malware recall is stuck at exactly 49%, just like the linear model.
* **Macro F1 (0.57):** Barely moved the needle. It's still completely failing to protect against the minority classes.

**Verdict:** Probabilistic models choke when features are correlated and data is non-linear. Both basic models (Linear & Naive Bayes) have officially failed to catch phishing. Time to bring in the heavy hitters: Tree-based models (Random Forest/XGBoost).

### XGBoost — RandomizedSearch
RandomizedSearch first to narrow down the parameter space — faster than full GridSearch on 640k rows. Then GridSearch refines the best region found here.

In [ ]:
param_dist = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [3, 6, 9],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier', XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        tree_method='hist',
        device='cuda',
        random_state=42
    ))
])
xgb_random = RandomizedSearchCV(
    xgb_pipeline,param_distributions=param_dist,
    n_iter = 5,cv=3,scoring='accuracy',verbose=2,n_jobs=-1,
    random_state=42
)
xgb_random.fit(X_train,y_train)
print(f'Best Parameters:{xgb_random.best_params_}')

In [ ]:
print("XGBoost Using Randomized Search")
y_pred_xgb = xgb_random.predict(X_test)
print(f'\n--Accuracy:{accuracy_score(y_test,y_pred_xgb)}')

print('\n---Confusion Matrix')
print(confusion_matrix(y_test,y_pred_xgb))

print('\n --Classification Report')
print(classification_report(y_test,y_pred_xgb))


### XGBoost(Randomized Search)

* **The Breakthrough (87% Accuracy & 0.81 Macro F1):** A massive jump from the 72-73% baseline. More importantly, the Macro F1 jumped from ~0.55 to 0.81, proving the model is actually learning the minority classes now, not just guessing 'Benign' to inflate accuracy.
* **The Phishing Leap (62% Recall):** This is the real victory. Linear models were stuck at 12-17% recall for phishing. XGBoost pulled it up to 62%. It is finally figuring out the complex, hidden non-linear splits to separate short, clean phishing URLs from legitimate ones. (Though 6,545 still slipped through as Safe, confirming our EDA that Phishing remains the hardest class to detect).
* **Malware Detection (74% Recall):** Jumped from 49% to 74% recall, with a phenomenal Precision of 98%. When XGBoost flags a URL as Malware, it is almost never wrong.
* **Defacement Detection (75% Recall):** Solid improvement, heavily utilizing the `query_length` and `special_char_count` features we identified earlier.

**Verdict:** XGBoost completely validates our hypothesis. URL security classification is inherently non-linear. The boosting architecture successfully uncovers the complex, overlapping structures of malicious URLs that simple algorithms just couldn't see.

##XGBoost (GridSearch)
Refining the best hyperparameter region found by RandomizedSearch to squeeze out maximum performance. We are locking in the best possible version of XGBoost here before pitting it against our final heavy-hitter: Random Forest. May the best tree win!

In [ ]:
# Grid search
param_grid_xgb = {
    'classifier__n_estimators': [40,50,70],
    'classifier__max_depth':[5,6,7],
    'classifier__learning_rate':[0.15,0.2,0.25]
}
xgb_grid = GridSearchCV(
    xgb_pipeline,param_grid=param_grid_xgb,
    cv=3,scoring='accuracy',
    verbose=2,
    n_jobs=-1
)
xgb_grid.fit(X_train,y_train)
print(f'Final Best Parameter:{xgb_grid.best_params_}')

In [ ]:
print("XGBoost Using Grid SearchCV")

grid_xgb_model = xgb_grid.best_estimator_

y_pred_xgb = grid_xgb_model.predict(X_test)

print(f'Accuracy:{accuracy_score(y_test,y_pred_xgb)}')

print('\n Confusion Matrix:')
print(confusion_matrix(y_test,y_pred_xgb))

print('\nClassification Report:')
print(classification_report(y_test,y_pred_xgb))


### XGBoost (GridSearch) Results: The Refined Benchmark

GridSearch successfully squeezed out extra performance by fine-tuning the best regions found by RandomizedSearch.

* **The GridSearch Bump:** Accuracy increased to 88.5%, and the Macro F1-score climbed from 0.81 to 0.84.
* **Malware Spike (82% Recall):** This is the biggest win. Malware recall jumped from 74% (in RandomizedSearch) to 82%, while maintaining a massive 97% precision. The model is now catching significantly more dangerous payloads without causing false alarms.
* **The Phishing Grind (66% Recall):** Phishing is still the toughest class, but we pushed its recall from 62% up to 66%.
* **Defacement (78% Recall):** A solid 3% bump from the previous iteration.

**Verdict:** GridSearch tightened the non-linear boundaries perfectly, making this our strongest model so far. Now, the stage is set. Let's bring in **Random Forest** and see if it can dethrone this highly-optimized XGBoost!

###Random Forest (GridSearch)

Our final model is a Random Forest ensemble. We are specifically utilizing the `class_weight='balanced'` parameter to directly counteract our highly imbalanced dataset (which is ~67% Benign). This mathematically forces the algorithm to penalize misclassifications in our minority classes (Phishing and Malware) more heavily.

The goal here is to see if this explicit class-weighting approach can improve minority recall compared to our optimized XGBoost model.

In [ ]:
print('Random Forest with Grid Search')
rf_pipeline = Pipeline([
    ('scaler',StandardScaler(with_mean=False)),
    ('classifier',RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])
param_grid = {
    'classifier__n_estimators':[50,100],
    'classifier__max_depth':[15,20],
    'classifier__min_samples_split':[5,10]}
rf_grid = GridSearchCV(rf_pipeline,
                       param_grid,cv=3,scoring='accuracy',
                       n_jobs=-1,
                       verbose=2)

rf_grid.fit(X_train,y_train)
print(f"Best Parameter:{rf_grid.best_params_}")

rf_pipeline.fit(X_train,y_train)
y_pred_rf = rf_grid.predict(X_test)

print(f'\nAccuracy:{accuracy_score(y_test,y_pred_rf)}')


print('\n Confusion Matrix:')
print(confusion_matrix(y_test,y_pred_rf))

print('\n Ctlassifcation Report:')
print(classification_report(y_test,y_pred_rf))

### Random Forest Results

* **Recall over Accuracy:** The overall accuracy (87.8%) is slightly lower than XGBoost (88.5%), but `class_weight='balanced'` did exactly what we needed. It shifted the model's focus to the minority classes.
* **Massive Jump in Threat Detection (Recall):**
    * Phishing jumped to 82% (up from 66%).
    * Malware hit 90% (up from 82%).
    * Defacement reached 88% (up from 78%).
* **The Trade-off:** To catch more threats, the model became more aggressive. This caused a drop in precision (e.g., it incorrectly flagged about 6,500 Benign URLs as Phishing).
* **Macro F1 (0.85):** Slightly better than XGBoost (0.84), proving it is mathematically more balanced across all four classes.

**Final Verdict:**
Random Forest is our final deployed model. In real-world cybersecurity, missing a malicious link (False Negative) can compromise a network, while mistakenly blocking a safe link (False Positive) is just a minor inconvenience. Random Forest prioritizes catching actual threats over looking good on paper, making it the right operational choice.

## Model Comparison

2 best models compared on same test set. XGBoost (GridSearch) wins on accuracy and macro F1 — chosen as deployment model.

In [ ]:
print("=== RANDOM FOREST ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf,
      target_names=['benign','phishing','defacement','malware']))

print("=== XGBOOST ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(classification_report(y_test, y_pred_xgb,
      target_names=['benign','phishing','defacement','malware']))

## Save Final Model

In [ ]:
joblib.dump(rf_grid.best_estimator_, 'url_model.pkl')
print('Saved: url_model.pkl')

joblib.dump(tld_freq, 'tld_freq.pkl')
print('Saved: tld_freq.pkl')

MODEL_COLS = list(rf_grid.best_estimator_.named_steps['scaler'].feature_names_in_)
joblib.dump(MODEL_COLS, 'model_cols.pkl')
print(f'Saved: model_cols.pkl → {MODEL_COLS}')

from google.colab import files
files.download('/content/url_model.pkl')
files.download('/content/tld_freq.pkl')
files.download('/content/model_cols.pkl')


###Live Inference & Sanity Check

Now that our Random Forest model is finalized, let's run a quick sanity check. We've built a custom prediction function that extracts features from raw URLs exactly as our training pipeline did.

We are testing it on a mix of standard benign URLs (Google, YouTube) and obvious malicious patterns (IP addresses, fake PayPal logins, double slashes) to see the probability breakdown and ensure the model behaves logically in real-time.

In [ ]:
def notebook_predict(url):
    f = extract_feature(url)
    ext = tldextract.extract(url)
    tld = ext.suffix.lower() if ext.suffix else 'no_tld'
    f['tld_encoded'] = tld_freq.get(tld, 0)
    df = pd.DataFrame([f])[MODEL_COLS]
    pred  = rf_grid.best_estimator_.predict(df)[0]
    proba = rf_grid.best_estimator_.predict_proba(df)[0]
    labels = {0:'benign',1:'phishing',2:'defacement',3:'malware'}
    print(f'URL: {url}')
    print(f' {labels[pred]} | B={proba[0]:.2f} Ph={proba[1]:.2f} D={proba[2]:.2f} M={proba[3]:.2f}')

test_urls = [
    'https://google.com',
    'https://www.youtube.com/watch?v=abc123',
    'https://www.microsoft.com/en-in',
    'http://paypal-secure-login.test/verify-account',
    'http://site.com//redirect/evil',
    'http://192.168.1.1/payload.exe',
]
for u in test_urls:
    notebook_predict(u)


###Inference Results: The Reality Check

Our manual tests reveal exactly how the model's learned rules apply in the real world—both the successes and the blind spots:

*   **The Length Paranoia (Google = Phishing):** The model flagged `google.com` as Phishing with 75% confidence. Why? Because we forced it to be aggressive (`class_weight='balanced'`), and our EDA proved that Phishing URLs are the shortest in the dataset. The model learned "extremely short and clean = phishing" a bit too well, causing a False Positive on a legitimate short domain.
*   **The Keyword Blindness (PayPal = Benign):** It completely missed the fake PayPal link. This perfectly validates our earlier EDA finding: `phishing_keyword` is a weak feature (present in only 10% of phishing data). The model ignored the text entirely, saw a normal-looking path length, and assumed it was a safe site.
*   **The IP Rule is Ironclad (192.168.1.1 = Malware):** It flagged the raw IP payload as Malware with **100% confidence**. It flawlessly applied the `has_ip` rule we identified as the strongest malware signal.
*   **Moderate Success on Standard Links:** It correctly classified YouTube and Microsoft as Benign, though the confidence levels (~45-50%) show it was slightly hesitant, likely due to the balanced weights making it naturally suspicious.

**Final Project Thought:**
This output is the perfect representation of real-world ML. The model excels at structural rules (catching IPs) but struggles with context (Google vs short phishing domains). To improve this further, we would need NLP features (like TF-IDF or BERT on the URL text) rather than relying strictly on numerical counts and lengths.